# Inspect Aggregated Soil Moisture Datasets

HRU-level inspection of the four soil moisture sources. Mirrors `inspect_consolidated_soil_moisture.ipynb`.

Sources (see `catalog/variables.yml` → `soil_moisture`):

- MERRA-2 (`GWETTOP`, dimensionless plant-available wetness, 0–5 cm).
- NLDAS-2 NOAH (`SoilM_0_10cm`, kg m⁻² in 0–10 cm).
- NLDAS-2 MOSAIC (`SoilM_0_10cm`, kg m⁻² in 0–10 cm).
- NCEP/NCAR R1 (`soilw_0_10cm`, **already volumetric water content** despite the upstream `kg/m2` mislabel).

See `docs/references/calibration-target-recipes.md` §4 for canonical-unit handling and the per-source 0–1 normalisation that combines them.

## Per-target conventions in this notebook

- **MERRA-2 `GWETTOP`** is plant-available wetness `(W − W_wilt) / (W_sat − W_wilt)`, **NOT volumetric water content**. Pass through unchanged. Layer 0–5 cm.
- **NLDAS-NOAH `SoilM_0_10cm`** is kg m⁻² in 0–10 cm. ÷ 100 → m³/m³ VWC.
- **NLDAS-MOSAIC `SoilM_0_10cm`** is kg m⁻² in 0–10 cm. ÷ 100 → m³/m³ VWC.
- **NCEP/NCAR `soilw_0_10cm`** is already VWC despite the upstream NetCDF labelling its `units` as `kg/m2` (recipes §4 — `valid_range` 0–1, `long_name` "Volumetric Soil Moisture", `actual_range` ~0.10–0.43 confirms VWC). **Pass through unchanged. Do NOT divide by 100.**
- Cross-source time alignment uses `select_month` (recipes lesson 2): MERRA-2 mid-month, NLDAS start-of-month, NCEP/NCAR end-of-month.
- The cross-source comparison plots in mixed units — MERRA-2's plant-available wetness sits in 0.1–0.9, VWC sits in 0.05–0.45. The target's per-source 0–1 normalisation cancels this offset; this notebook displays the unnormalised values so the constant offset is visible.
- Reference source for the colour scale: NLDAS-NOAH (CONUS-only, 0.125°).

In [ ]:
import calendar
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    discover_aggregated,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    select_month,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/inspect_aggregated/<run>_*.png
# import _helpers
# _helpers.SAVE_FIGURES = True

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

TARGET = "soil_moisture"
TARGET_TIME = "2000-01-15"
TARGET_YEAR = 2000
TARGET_MONTH = 1
TIME_SERIES_YEARS = range(2000, 2011)
REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Phoenix metro (arid SW)": (-112.1, 33.4),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}
SOURCES = {
    "merra2": {"label": "MERRA-2 (GWETTOP)", "var": "GWETTOP"},
    "nldas_noah": {"label": "NLDAS-2 NOAH (SoilM_0_10cm)", "var": "SoilM_0_10cm"},
    "nldas_mosaic": {"label": "NLDAS-2 MOSAIC (SoilM_0_10cm)", "var": "SoilM_0_10cm"},
    "ncep_ncar": {"label": "NCEP/NCAR R1 (soilw_0_10cm)", "var": "soilw_0_10cm"},
}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")


## Load aggregated datasets

Each source is opened from `<project>/data/aggregated/<source_key>/<source_key>_<TARGET_YEAR>_agg.nc`. Sources whose aggregation has not yet been produced are skipped with a clear message; downstream cells iterate over the loaded set so missing sources drop out naturally.

In [ ]:
opened = {}
for source_key, info in SOURCES.items():
    paths = discover_aggregated(project_dir, source_key)
    if paths is None:
        print(f"SKIP {info['label']}: no aggregated NCs at "
              f"{project_dir}/data/aggregated/{source_key}/")
        continue
    ds = open_year(project_dir, source_key, TARGET_YEAR)
    info["units"] = unit_from_catalog(source_key, info["var"])
    opened[source_key] = (ds, info)
    values = ds[info["var"]].isel(time=0).to_pandas()
    print(
        f"{info['label']}: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get('nhm_id', ds.sizes.get('hru_id', 'unknown'))} | "
        f"NaN HRUs (t=0): {nan_hru_count(values)} | "
        f"catalog units: {info['units']}"
    )


In [ ]:
for source_key, (ds, info) in opened.items():
    print(f"{'=' * 60}\n{info['label']}\n{'=' * 60}")
    display(ds)


In [ ]:
if not opened:
    print("No sources available; skipping native-unit map.")
else:
    fig, axes = plt.subplots(1, len(opened), figsize=(8 * len(opened), 6), squeeze=False)
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        values = da.to_pandas()
        plot_hru_choropleth(
            axes[0, idx],
            fabric,
            values,
            cmap="YlGnBu",
            title=f"{info['label']}\n{TARGET_TIME} | {info['units']}",
            units=info["units"],
        )
    fig.suptitle(f"Soil Moisture — native units, {TARGET_TIME}", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_native_units_map")
    plt.show()


In [ ]:
plt.close("all")
import gc; gc.collect()



## Normalized comparison map (mixed dimensionless / m³ m⁻³)

Apply per-source unit handling and plot side by side on a shared colour scale derived from NLDAS-NOAH (the CONUS-only reference source).

- MERRA-2 `GWETTOP` — plant-available wetness, dimensionless. Pass through.
- NLDAS-NOAH `SoilM_0_10cm` — kg m⁻² in 0–10 cm. ÷ 100 → m³/m³ VWC.
- NLDAS-MOSAIC `SoilM_0_10cm` — same as NOAH; ÷ 100 → m³/m³ VWC.
- NCEP/NCAR `soilw_0_10cm` — already VWC despite the on-disk `kg/m2` mislabel. Pass through.

The map plots all four sources at once. MERRA-2 sits in 0.1–0.9 (plant-available); the others sit in 0.05–0.45 (VWC). The target's per-source 0–1 normalisation cancels that offset; here we deliberately show the unnormalised values so the offset is visible.

In [ ]:
def _to_vwc_or_passthrough(da: xr.DataArray, source_key: str) -> xr.DataArray:
    if source_key == "merra2":
        return da  # plant-available wetness; passthrough
    if source_key in {"nldas_noah", "nldas_mosaic"}:
        return da / 100.0  # kg m-2 / (0.10 m × 1000 kg/m³) = m³/m³
    if source_key == "ncep_ncar":
        return da  # already VWC despite the kg/m2 mislabel; recipes §4
    raise ValueError(f"No SOM conversion for {source_key}")


if opened:
    converted = {}
    for source_key, (ds, info) in opened.items():
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        converted[source_key] = _to_vwc_or_passthrough(da, source_key).to_pandas()

    ref_key = "nldas_noah"
    if ref_key in converted:
        ref_vals = converted[ref_key].dropna().values
        vmin, vmax = float(np.percentile(ref_vals, 2)), float(np.percentile(ref_vals, 98))
    else:
        all_vals = np.concatenate([s.dropna().values for s in converted.values()])
        vmin, vmax = float(np.percentile(all_vals, 2)), float(np.percentile(all_vals, 98))

    fig, axes = plt.subplots(1, len(converted), figsize=(8 * len(converted), 6), squeeze=False)
    for idx, (source_key, values) in enumerate(converted.items()):
        plot_hru_choropleth(
            axes[0, idx],
            fabric,
            values,
            vmin=vmin,
            vmax=vmax,
            cmap="YlGnBu",
            title=f"{SOURCES[source_key]['label']}\n{TARGET_TIME} | (dimensionless / m³ m⁻³)",
            units="(dimensionless / m³ m⁻³)",
        )
    fig.suptitle(
        f"Soil Moisture — colour scale from NLDAS-NOAH — {TARGET_TIME}",
        fontsize=13, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_normalized_comparison")
    plt.show()


In [ ]:
if opened:
    fig, ax = plt.subplots(figsize=(10, 5))
    for source_key, values in converted.items():
        ax.hist(
            values.dropna(),
            bins=60,
            histtype="step",
            label=SOURCES[source_key]["label"],
            density=True,
            linewidth=2,
        )
    ax.set_xlabel("Soil Moisture (dimensionless / m³ m⁻³)")
    ax.set_ylabel("Density")
    ax.set_title(f"Cross-source HRU distribution — {TARGET_TIME}")
    ax.legend()
    save_figure(fig, f"{TARGET}_histogram")
    plt.show()


In [ ]:
if opened:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    series_data = {}
    for source_key, info in SOURCES.items():
        if source_key not in opened:
            continue
        ds_range = open_year_range(project_dir, source_key, TIME_SERIES_YEARS)
        try:
            id_dim = "nhm_id" if "nhm_id" in ds_range.dims else "hru_id"
            sel = ds_range[info["var"]].sel({id_dim: list(rep_hrus.values())}).load()
        finally:
            ds_range.close()
        series_data[source_key] = sel

    def _convert_series(da, source_key):
        """Apply per-source SOM unit handling. Expects 1-D (time,) input — select an HRU first."""
        if source_key == "merra2":
            return da
        if source_key in {"nldas_noah", "nldas_mosaic"}:
            return da / 100.0
        if source_key == "ncep_ncar":
            return da
        return da

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    id_dim = "nhm_id" if "nhm_id" in next(iter(series_data.values())).dims else "hru_id"
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        for source_key, da in series_data.items():
            da_hru = _convert_series(da.sel({id_dim: hru_id}), source_key)
            ax.plot(da_hru.time, da_hru.values, label=SOURCES[source_key]["label"])
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("(dimensionless / m³ m⁻³)")
        ax.legend(fontsize=8)
    fig.suptitle(
        f"Soil Moisture at representative HRUs — {min(TIME_SERIES_YEARS)}–{max(TIME_SERIES_YEARS)}"
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_time_series")
    plt.show()


In [ ]:
if opened:
    fig, axes = plt.subplots(1, len(opened), figsize=(8 * len(opened), 6), squeeze=False)
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        values = da.to_pandas()
        n_nan = nan_hru_count(values)
        print(f"{info['label']}: {n_nan} NaN HRUs ({100 * n_nan / len(fabric):.2f}%)")
        plot_nan_hrus(
            axes[0, idx],
            fabric,
            values,
            title=f"{info['label']}\nNaN HRUs (red) — {n_nan} of {len(fabric)}",
        )
    fig.suptitle(
        "Coverage gaps — these will be nearest-neighbor-filled in normalize/ before target combination",
        fontsize=12,
        y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_coverage")
    plt.show()


In [ ]:
def _gridded_mean_som(source_key, info):
    """Compute the gridded CONUS-footprint mean for the validation cell.

    Approximate cross-check, not a like-for-like comparison: gridded mean
    is unweighted over the consolidated-NC bbox; HRU mean is Albers-area-
    weighted over fabric HRUs only. Differences of 5-30% are normal.
    """
    if source_key == "merra2":
        path = datastore_dir / "merra2" / "merra2_monthly.nc"
    elif source_key == "nldas_noah":
        path = datastore_dir / "nldas_noah" / "nldas_noah_monthly.nc"
    elif source_key == "nldas_mosaic":
        path = datastore_dir / "nldas_mosaic" / "nldas_mosaic_monthly.nc"
    elif source_key == "ncep_ncar":
        path = datastore_dir / "ncep_ncar" / "ncep_ncar_monthly.nc"
    else:
        return None, f"unknown gridded path for {source_key}"
    if not path.exists():
        return None, f"missing consolidated NC at {path}"
    with xr.open_dataset(path) as ds:
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH).load()
    return float(_to_vwc_or_passthrough(da, source_key).mean(skipna=True).item()), None


print(f"{'Source':<35} {'Aggregated':>12} {'Gridded':>12} {'Δ':>12} {'% diff':>8}")
print("-" * 85)
for source_key, (ds, info) in opened.items():
    da_agg = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
    converted_agg = _to_vwc_or_passthrough(da_agg, source_key).to_pandas()
    agg_mean = area_weighted_mean(converted_agg, fabric)
    gridded_mean, reason = _gridded_mean_som(source_key, info)
    if gridded_mean is None:
        print(f"{info['label']:<35} {agg_mean:>12.4f}  {'SKIP':>12} ({reason})")
        continue
    diff = agg_mean - gridded_mean
    pct = 100 * diff / gridded_mean if gridded_mean else float("nan")
    print(f"{info['label']:<35} {agg_mean:>12.4f} {gridded_mean:>12.4f} {diff:>12.4f} {pct:>7.2f}%")


## Why HRU-level patterns differ across sources

After area-weighted aggregation to HRUs, the HRU-level magnitudes are within the same order of magnitude as the gridded means (validation cell above). The validation cell's "gridded" column is an unweighted mean over the consolidated-NC bbox; the "aggregated" column is an Albers-area-weighted mean over fabric HRUs only. A 5–30% difference between the two columns is normal.

The four sources measure related but not identical quantities:

- **MERRA-2 GWETTOP** — plant-available wetness fraction `(W − W_wilt) / (W_sat − W_wilt)` in the 0–5 cm layer. Typical CONUS values 0.1–0.9.
- **NLDAS-2 NOAH/MOSAIC** — VWC in 0–10 cm after the `÷ 100` conversion. Typical 0.05–0.45.
- **NCEP/NCAR R1 `soilw_0_10cm`** — VWC in 0–10 cm (pass-through; the `kg/m2` label is wrong). Typical 0.10–0.43.

Layer depths and definitions differ enough that absolute-value comparison is misleading; the target's per-source 0–1 normalisation handles this. This notebook shows unnormalised values because that's where unit-conversion mistakes are visible.

**Calibration target implication.** Mistakes that look like silent failures here:

- Dividing NCEP/NCAR by 100 produces values around 0.001–0.005 — physically impossible VWC.
- Treating MERRA-2 GWETTOP as VWC and converting via layer thickness gives mm-depth-of-water values that look wrong by a factor of layer thickness.
- Reading `units` from the NetCDF instead of the catalog (lesson 9) produces inconsistent treatment as soon as a source is re-fetched but the catalog is corrected.

The validation cell catches all three: the % diff against the gridded mean explodes if any of them is mis-applied.

In [ ]:
for source_key, (ds, _) in opened.items():
    ds.close()
opened.clear()
